<a href="https://colab.research.google.com/github/eduardo-r-carotenuto/Metro-Conect/blob/main/C%C3%B3pia_de_C%C3%B3pia_de_Desafio_MetroBot_SP_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MetrôBot SP 2.0 — Desafio (Linhas 1, 2 e 3)

**Curso:** _(Ciência da Computação)_
**Integrantes:** _(Daniel Santiago, Eduardo Carotenuto, Everton Tiburcio, Fauzer Ribeiro e Matheus Diorio)_

**Como rodar:**
1. Rode as células em ordem, de cima para baixo (ou "Executar tudo" / "Run All").
2. Por padrão `PROVEDOR = "offline"` (célula 2), então o notebook funciona **sem internet e sem chave de API**.
3. Para usar o Llama de verdade, troque `PROVEDOR` para `"groq"` (e configure `GROQ_API_KEY` nos Secrets do Colab ou num `.env`) ou `"ollama"`.
4. Ao final há `rodar_testes()` com os 6 casos obrigatórios + 4 extras, e o painel interativo com ipywidgets.

Este notebook reaproveita tudo que foi construído na Aula Prática 01 (BFS, DFS, lógica proposicional e de primeira ordem, motor de inferência, intérprete/narrador com Llama e modo offline) e estende para as **3 linhas do metrô**, com baldeações, integrações deduzidas automaticamente (R6) e uma regra nova criada pelo grupo (R7).

In [ ]:
# Célula 1 — instalação (rode uma vez)
%pip install -q groq ollama ipywidgets python-dotenv

In [ ]:
# Célula 2 — configuração e função única de LLM (igual à Aula 01)
import os, json, re, unicodedata
from collections import deque, defaultdict
from itertools import product

PROVEDOR = "groq" # "groq" (nuvem), "ollama" (local) ou "offline" (sem LLM)
MODELO_GROQ = "openai/gpt-oss-20b"
MODELO_OLLAMA = "llama3.2"

def obter_chave_groq():
    """Busca a chave SEM escrevê-la no código: Colab Secrets → .env → variável de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    return os.environ.get("GROQ_API_KEY")

def chamar_llm(mensagens, modo_json=False):
    """Envia mensagens ao Llama e devolve o texto da resposta."""
    if PROVEDOR == "groq":
        from groq import Groq
        cliente = Groq(api_key=obter_chave_groq())
        extras = {"response_format": {"type": "json_object"}} if modo_json else {}
        resposta = cliente.chat.completions.create(
            model=MODELO_GROQ, messages=mensagens, temperature=0, **extras)
        return resposta.choices[0].message.content
    elif PROVEDOR == "ollama":
        import ollama
        extras = {"format": "json"} if modo_json else {}
        resposta = ollama.chat(model=MODELO_OLLAMA, messages=mensagens,
                                options={"temperature": 0}, **extras)
        return resposta["message"]["content"]
    else:
        raise RuntimeError("Modo offline: nenhum LLM configurado.")

## R1 — Modelagem: grafo das 3 linhas

Cada estação é um **nó único**, mesmo quando pertence a mais de uma linha (ex.: "Sé" na Linha 1 e na Linha 3 é o mesmo nó — é isso que conecta as linhas). Guardamos também `linhas_do_trecho`, indicando quais linhas passam por cada trecho (algumas, como Paraíso–Ana Rosa, pertencem a duas linhas ao mesmo tempo).

In [ ]:
# Célula 3 — dados das 3 linhas (use exatamente estes nomes)
LINHAS = {
    "Linha 1-Azul": [
        "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
        "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
        "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
        "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
        "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
    ],
    "Linha 2-Verde": [
        "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
        "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",
        "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã", "Tamanduateí",
        "Vila Prudente",
    ],
    "Linha 3-Vermelha": [
        "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",
        "República", "Anhangabaú", "Sé", "Pedro II", "Brás",
        "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",
        "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",
        "Artur Alvim", "Corinthians-Itaquera",
    ],
}

CORES = {"Linha 1-Azul": "#1e88e5", "Linha 2-Verde": "#2e7d32",
         "Linha 3-Vermelha": "#d32f2f"}

In [ ]:
# Célula 4 — R1: construir_grafo_multilinhas
def construir_grafo_multilinhas(linhas):
    """Retorna (grafo, linhas_do_trecho).
    grafo: {estacao: [vizinhas sem repetição]}
    linhas_do_trecho: {(a,b): {nomes das linhas que passam por esse trecho}}"""
    grafo = {}
    linhas_do_trecho = {}
    for nome_linha, estacoes in linhas.items():
        for estacao in estacoes:
            grafo.setdefault(estacao, [])
        for i in range(len(estacoes) - 1):
            a, b = estacoes[i], estacoes[i + 1]
            if b not in grafo[a]:
                grafo[a].append(b)
            if a not in grafo[b]:
                grafo[b].append(a)
            linhas_do_trecho.setdefault((a, b), set()).add(nome_linha)
            linhas_do_trecho.setdefault((b, a), set()).add(nome_linha)
    return grafo, linhas_do_trecho

GRAFO, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)

# Conferindo
print("Total de estações (esperado 52):", len(GRAFO))
sem_duplicatas = all(len(v) == len(set(v)) for v in GRAFO.values())
print("Nenhum vizinho duplicado:", sem_duplicatas)
print("Trecho Paraíso→Ana Rosa pertence a:", LINHAS_DO_TRECHO[("Paraíso", "Ana Rosa")])
print("Vizinhas da Sé (integração 1-Azul / 3-Vermelha):", GRAFO["Sé"])

Total de estações (esperado 52): 52
Nenhum vizinho duplicado: True
Trecho Paraíso→Ana Rosa pertence a: {'Linha 1-Azul', 'Linha 2-Verde'}
Vizinhas da Sé (integração 1-Azul / 3-Vermelha): ['São Bento', 'Japão-Liberdade', 'Anhangabaú', 'Pedro II']


## R2 — Busca (BFS e DFS) + contagem de baldeações

As funções `bfs` e `dfs` são as mesmas da Aula 01 (funcionam em qualquer grafo, já respeitando `bloqueadas`). O que é novo aqui é `contar_baldeacoes`, que percorre o caminho encontrado e identifica em quais estações é preciso trocar de linha (mantendo a linha atual sempre que ela ainda servir para o próximo trecho).

In [ ]:
# Célula 5 — BFS e DFS (iguais à Aula 01, genéricas para qualquer grafo)
def reconstruir_caminho(pai, destino):
    caminho = []
    atual = destino
    while atual is not None:
        caminho.append(atual)
        atual = pai[atual]
    return list(reversed(caminho))

def bfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft()
        ordem_visita.append(atual)
        if atual == destino:
            return reconstruir_caminho(pai, destino), ordem_visita
        for vizinho in grafo[atual]:
            if vizinho not in pai and vizinho not in bloqueadas:
                pai[vizinho] = atual
                fila.append(vizinho)
    return None, ordem_visita

def dfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    visitados = set()
    ordem_visita = []
    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino:
            return caminho
        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho])
                if resultado:
                    return resultado
        return None
    return explorar(origem, [origem]), ordem_visita

In [ ]:
# Célula 6 — contagem de baldeações
def contar_baldeacoes(caminho, linhas_do_trecho):
    """Retorna (quantidade, [(estacao, linha_nova), ...]).
    Percorre os trechos do caminho mantendo a 'linha atual'; só troca de linha
    quando ela realmente não serve mais para o próximo trecho."""
    if not caminho or len(caminho) < 2:
        return 0, []
    baldeacoes = []
    linha_atual = None
    for i in range(len(caminho) - 1):
        a, b = caminho[i], caminho[i + 1]
        linhas_trecho = linhas_do_trecho[(a, b)]
        if linha_atual is None or linha_atual not in linhas_trecho:
            nova_linha = sorted(linhas_trecho)[0]
            if linha_atual is not None:
                baldeacoes.append((a, nova_linha))
            linha_atual = nova_linha
    return len(baldeacoes), baldeacoes

In [ ]:
# Célula 7 — comparando BFS vs. DFS em algumas viagens (esforço de cada algoritmo)
viagens = [("Tucuruvi", "Corinthians-Itaquera"), ("Vila Madalena", "Jabaquara"),
           ("Palmeiras-Barra Funda", "Vila Prudente")]

print(f"{'Viagem':<38}{'Paradas':>8}{'Baldeações':>12}{'BFS visitou':>13}{'DFS visitou':>13}")
for origem, destino in viagens:
    c_bfs, v_bfs = bfs(GRAFO, origem, destino)
    c_dfs, v_dfs = dfs(GRAFO, origem, destino)
    n_bald, _ = contar_baldeacoes(c_bfs, LINHAS_DO_TRECHO)
    print(f"{origem + ' → ' + destino:<38}{len(c_bfs)-1:>8}{n_bald:>12}{len(v_bfs):>13}{len(v_dfs):>13}")

Viagem                                 Paradas  Baldeações  BFS visitou  DFS visitou
Tucuruvi → Corinthians-Itaquera             22           1           52           52
Vila Madalena → Jabaquara                   14           1           37           46
Palmeiras-Barra Funda → Vila Prudente       16           2           49           34


## R3 — Base de conhecimento e lógica (R1–R7)

- **Locais conhecidos:** pelo menos 3 por linha (13 na Linha 1, 3 na Linha 2, 3 na Linha 3 — 19 no total).
- **R1–R5**: as mesmas regras da Aula 01 (origem, destino, bloqueio, acessibilidade, alerta), agora operando sobre as 52 estações.
- **R6 (obrigatória — integração):** `∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1≠l2 → integracao(e))`. As integrações (Sé, Paraíso, Ana Rosa) são **deduzidas pelo motor de inferência**, nunca digitadas à mão.
- **R7 (criada pelo grupo — linha paralisada):** `∀e ∀l (linha_paralisada(l) ∧ pertence(e,l) → bloqueada(e))`. Se uma linha inteira for marcada como paralisada (ex.: greve simulada), **todas** as suas estações ficam bloqueadas para a busca.

In [ ]:
# Célula 8 — locais conhecidos (mínimo 3 por linha)
LOCAIS = {
    # Linha 1-Azul
    "Shopping Metrô Tucuruvi": "Tucuruvi",
    "Terminal Rodoviário Tietê": "Portuguesa-Tietê",
    "Museu de Arte Sacra": "Tiradentes",
    "Pinacoteca": "Luz",
    "Museu da Língua Portuguesa": "Luz",
    "Mosteiro de São Bento": "São Bento",
    "Rua 25 de Março": "São Bento",
    "Catedral da Sé": "Sé",
    "Bairro da Liberdade": "Japão-Liberdade",
    "Centro Cultural São Paulo": "Vergueiro",
    "Shopping Metrô Santa Cruz": "Santa Cruz",
    "Universidade São Judas": "São Judas",
    "Terminal Rodoviário Jabaquara": "Jabaquara",
    # Linha 2-Verde
    "MASP": "Trianon-Masp",
    "Hospital das Clínicas": "Clínicas",
    "Sesc Consolação": "Consolação",
    # Linha 3-Vermelha
    "Theatro Municipal": "Anhangabaú",
    "Neo Química Arena": "Corinthians-Itaquera",
    "Shopping Metrô Tatuapé": "Tatuapé",
}
TODAS_ESTACOES = sorted(GRAFO.keys())
print("Locais conhecidos:", len(LOCAIS), "| Estações:", len(TODAS_ESTACOES))

Locais conhecidos: 19 | Estações: 52


In [ ]:
# Célula 9 — fatos e consulta (lógica de primeira ordem)
def fatos_base():
    """Fatos fixos: quais estações existem, a quais linhas pertencem e o que fica perto de cada uma."""
    fatos = set()
    for nome_linha, estacoes in LINHAS.items():
        for estacao in estacoes:
            fatos.add(("estacao", estacao))
            fatos.add(("pertence", estacao, nome_linha))
    for local, estacao in LOCAIS.items():
        fatos.add(("proximo_de", local, estacao))
    return fatos

def consultar(fatos, predicado):
    """Devolve os argumentos de todos os fatos de um predicado."""
    return [f[1:] for f in fatos if f[0] == predicado]

In [ ]:
# Célula 10 — regras R1 a R7
def r_origem(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_esta_em"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("origem", e))
    for (e,) in consultar(fatos, "usuario_esta_na_estacao"):
        novos.add(("origem", e))
    return novos

def r_destino(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_quer_ir"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("destino", e))
    for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):
        novos.add(("destino", e))
    return novos

def r_bloqueio(fatos):
    return {("bloqueada", e) for (e,) in consultar(fatos, "fechada")}

def r_acessibilidade(fatos):
    if not consultar(fatos, "precisa_acessibilidade"):
        return set()
    return {("inacessivel", e) for (e,) in consultar(fatos, "elevador_em_manutencao")}

def r_alerta(fatos):
    novos = set()
    inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}
    for papel in ("origem", "destino"):
        for (e,) in consultar(fatos, papel):
            if e in inacessiveis:
                novos.add(("alerta", papel, e))
    return novos

def r_integracao(fatos):
    """R6 (obrigatória): ∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1≠l2 → integracao(e))"""
    por_estacao = defaultdict(set)
    for e, l in consultar(fatos, "pertence"):
        por_estacao[e].add(l)
    return {("integracao", e) for e, linhas in por_estacao.items() if len(linhas) > 1}

def r_linha_paralisada(fatos):
    """R7 (criada pelo grupo): ∀e ∀l (linha_paralisada(l) ∧ pertence(e,l) → bloqueada(e))
    Se uma linha inteira estiver paralisada (ex.: greve simulada), todas as suas estações ficam bloqueadas."""
    paralisadas = {l for (l,) in consultar(fatos, "linha_paralisada")}
    if not paralisadas:
        return set()
    return {("bloqueada", e) for e, l in consultar(fatos, "pertence") if l in paralisadas}

REGRAS = [
    ("R1 origem", "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))", r_origem),
    ("R2 destino", "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))", r_destino),
    ("R3 bloqueio", "∀e (fechada(e) → bloqueada(e))", r_bloqueio),
    ("R4 acessibilidade", "∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))", r_acessibilidade),
    ("R5 alerta", "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))", r_alerta),
    ("R6 integracao", "∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1≠l2 → integracao(e))", r_integracao),
    ("R7 linha_paralisada", "∀e ∀l (linha_paralisada(l) ∧ pertence(e,l) → bloqueada(e))", r_linha_paralisada),
]

def encadear_para_frente(fatos, regras, verbose=False):
    """Aplica as regras em rodadas até não surgir nenhum fato novo (ponto fixo)."""
    fatos = set(fatos)
    justificativas = {}
    rodada = 0
    while True:
        rodada += 1
        novos_na_rodada = set()
        for nome, _formula, regra in regras:
            for fato in regra(fatos) - fatos:
                novos_na_rodada.add(fato)
                justificativas[fato] = nome
        if verbose:
            print(f"Rodada {rodada}: {len(novos_na_rodada)} fato(s) novo(s)")
        if not novos_na_rodada:
            return fatos, justificativas
        fatos |= novos_na_rodada

In [ ]:
# Célula 11 — vendo a R6 deduzir as integrações sozinha (sem digitar à mão)
fatos_demo, justificativas_demo = encadear_para_frente(fatos_base(), REGRAS, verbose=True)
integracoes = sorted({e for (e,) in consultar(fatos_demo, "integracao")})
print("\nEstações de integração deduzidas pela R6:", integracoes)
print("(esperado: ['Ana Rosa', 'Paraíso', 'Sé'])")

Rodada 1: 3 fato(s) novo(s)
Rodada 2: 0 fato(s) novo(s)

Estações de integração deduzidas pela R6: ['Ana Rosa', 'Paraíso', 'Sé']
(esperado: ['Ana Rosa', 'Paraíso', 'Sé'])


In [ ]:
# Célula 12 — tabela-verdade de uma regra proposicional à escolha do grupo
# P: estação aberta | Q: precisa de acessibilidade | R: elevador funcionando | S: linha paralisada (R7)
def pode_embarcar_v2(P, Q, R, S):
    return P and ((not Q) or R) and (not S)

def tabela_verdade_v2():
    print(f" {'P':5} | {'Q':5} | {'R':5} | {'S':5} | resultado")
    print("-" * 45)
    for P, Q, R, S in product([True, False], repeat=4):
        print(f" {P!s:5} | {Q!s:5} | {R!s:5} | {S!s:5} | {pode_embarcar_v2(P, Q, R, S)}")

tabela_verdade_v2()

 P     | Q     | R     | S     | resultado
---------------------------------------------
 True  | True  | True  | True  | False
 True  | True  | True  | False | True
 True  | True  | False | True  | False
 True  | True  | False | False | False
 True  | False | True  | True  | False
 True  | False | True  | False | True
 True  | False | False | True  | False
 True  | False | False | False | True
 False | True  | True  | True  | False
 False | True  | True  | False | False
 False | True  | False | True  | False
 False | True  | False | False | False
 False | False | True  | True  | False
 False | False | True  | False | False
 False | False | False | True  | False
 False | False | False | False | False


## O planejador (lógica + busca + baldeações)

In [ ]:
# Célula 13 — planejar(): monta fatos, roda a inferência e chama a busca
TEMPO_POR_TRECHO = 2   # min por trecho (simulado)
TEMPO_BALDEACAO = 5    # min extra por troca de linha (simulado)

def planejar(pedido, fechadas=(), manutencao=(), linhas_paralisadas=(), algoritmo="BFS"):
    """pedido = {"origem": (tipo, nome), "destino": (tipo, nome), "acessibilidade": bool}"""
    fatos = fatos_base()
    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]
    fatos.add(("usuario_esta_em", nome_o) if tipo_o == "local" else ("usuario_esta_na_estacao", nome_o))
    fatos.add(("usuario_quer_ir", nome_d) if tipo_d == "local" else ("usuario_quer_ir_estacao", nome_d))
    if pedido.get("acessibilidade"):
        fatos.add(("precisa_acessibilidade",))
    for e in fechadas:
        fatos.add(("fechada", e))
    for e in manutencao:
        fatos.add(("elevador_em_manutencao", e))
    for l in linhas_paralisadas:
        fatos.add(("linha_paralisada", l))

    fatos, justificativas = encadear_para_frente(fatos, REGRAS)

    origem = consultar(fatos, "origem")[0][0]
    destino = consultar(fatos, "destino")[0][0]
    bloqueadas = {e for (e,) in consultar(fatos, "bloqueada")}
    alertas = consultar(fatos, "alerta")
    integracoes = {e for (e,) in consultar(fatos, "integracao")}

    buscar = bfs if algoritmo == "BFS" else dfs
    caminho, visitados = buscar(GRAFO, origem, destino, bloqueadas)

    n_baldeacoes, pontos_baldeacao = (contar_baldeacoes(caminho, LINHAS_DO_TRECHO)
                                       if caminho else (None, []))
    tempo = None
    if caminho:
        tempo = (len(caminho) - 1) * TEMPO_POR_TRECHO + n_baldeacoes * TEMPO_BALDEACAO

    return {
        "origem": origem, "destino": destino, "algoritmo": algoritmo,
        "caminho": caminho, "visitados": visitados,
        "bloqueadas": sorted(bloqueadas),
        "alertas": [f"{p}: {e}" for p, e in alertas],
        "integracoes_no_caminho": [e for e in (caminho or []) if e in integracoes],
        "paradas": len(caminho) - 1 if caminho else None,
        "baldeacoes": n_baldeacoes,
        "pontos_baldeacao": pontos_baldeacao,
        "tempo_min": tempo,
        "regras_usadas": sorted(set(justificativas.values())),
    }

In [ ]:
# Célula 14 — testando o planejador
r = planejar({"origem": ("local", "Catedral da Sé"), "destino": ("local", "Neo Química Arena")})
for chave, valor in r.items():
    print(f"{chave:>22}: {valor}")

                origem: Sé
               destino: Corinthians-Itaquera
             algoritmo: BFS
               caminho: ['Sé', 'Pedro II', 'Brás', 'Bresser-Mooca', 'Belém', 'Tatuapé', 'Carrão', 'Penha', 'Vila Matilde', 'Guilhermina-Esperança', 'Patriarca-Vila Ré', 'Artur Alvim', 'Corinthians-Itaquera']
             visitados: ['Sé', 'São Bento', 'Japão-Liberdade', 'Anhangabaú', 'Pedro II', 'Luz', 'São Joaquim', 'República', 'Brás', 'Tiradentes', 'Vergueiro', 'Santa Cecília', 'Bresser-Mooca', 'Armênia', 'Paraíso', 'Marechal Deodoro', 'Belém', 'Portuguesa-Tietê', 'Ana Rosa', 'Brigadeiro', 'Palmeiras-Barra Funda', 'Tatuapé', 'Carandiru', 'Vila Mariana', 'Chácara Klabin', 'Trianon-Masp', 'Carrão', 'Santana', 'Santa Cruz', 'Santos-Imigrantes', 'Consolação', 'Penha', 'Jardim São Paulo', 'Praça da Árvore', 'Alto do Ipiranga', 'Clínicas', 'Vila Matilde', 'Parada Inglesa', 'Saúde', 'Sacomã', 'Sumaré', 'Guilhermina-Esperança', 'Tucuruvi', 'São Judas', 'Tamanduateí', 'Vila Madalena', 'Patriar

## R4 — Llama: intérprete e narrador (com guardrails e modo offline)

In [ ]:
# Célula 15 — intérprete (Llama) com validação e fallback offline
def normalizar(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def resolver_nome(nome):
    """GUARDRAIL: só aceita nomes que existem de verdade nas 52 estações ou nos locais. Senão, None."""
    if not nome:
        return None
    alvo = normalizar(nome).strip()
    for estacao in TODAS_ESTACOES:
        if normalizar(estacao) == alvo:
            return ("estacao", estacao)
    for local in LOCAIS:
        if normalizar(local) == alvo:
            return ("local", local)
    return None

PROMPT_INTERPRETE = """Você é o módulo de INTERPRETAÇÃO do MetrôBot SP (Linhas 1, 2 e 3).
Sua única tarefa é transformar o pedido do passageiro em JSON.

Estações válidas: {estacoes}
Locais válidos: {locais}

Responda APENAS com um JSON neste formato:
{{"origem": "<nome exato de estação ou local, ou null>",
  "destino": "<nome exato de estação ou local, ou null>",
  "acessibilidade": <true ou false>}}

Regras:
- Use SOMENTE nomes das listas acima, escritos exatamente como aparecem.
- "acessibilidade" é true se o passageiro mencionar cadeira de rodas,
  mobilidade reduzida, muletas, carrinho de bebê ou precisar de elevador.
- Se não souber algum campo, use null. Nunca invente nomes."""

def interpretar_offline(texto):
    """Plano B sem LLM: procura nomes conhecidos no texto, na ordem em que aparecem."""
    texto_min = texto.lower()
    texto_sem = normalizar(texto)
    candidatos = [(n, "estacao") for n in TODAS_ESTACOES] + [(n, "local") for n in LOCAIS]
    candidatos.sort(key=lambda c: len(c[0]), reverse=True)
    ocupado = [False] * len(texto_min)
    encontrados = []
    for nome, tipo in candidatos:
        buscas = [(texto_min, nome.lower())]
        if len(nome) > 4 and len(texto_sem) == len(texto_min):
            buscas.append((texto_sem, normalizar(nome)))
        for base, padrao in buscas:
            for m in re.finditer(r"(?<!\w)" + re.escape(padrao) + r"(?!\w)", base):
                if not any(ocupado[m.start():m.end()]):
                    encontrados.append((m.start(), nome))
                    for i in range(m.start(), m.end()):
                        ocupado[i] = True
    encontrados.sort()
    palavras_acess = ["cadeira de rodas", "acessibilidade", "mobilidade",
                       "muleta", "carrinho de bebe", "elevador"]
    return {
        "origem": encontrados[0][1] if len(encontrados) > 0 else None,
        "destino": encontrados[1][1] if len(encontrados) > 1 else None,
        "acessibilidade": any(p in texto_sem for p in palavras_acess),
    }

def interpretar_pedido(texto):
    """Texto livre → pedido validado. Usa o Llama; se falhar, cai no modo offline."""
    if PROVEDOR == "offline":
        bruto = interpretar_offline(texto)
        fonte = "offline"
    else:
        sistema = PROMPT_INTERPRETE.format(
            estacoes=", ".join(TODAS_ESTACOES), locais=", ".join(LOCAIS))
        try:
            resposta = chamar_llm([{"role": "system", "content": sistema},
                                    {"role": "user", "content": texto}], modo_json=True)
            bruto = json.loads(resposta)
            fonte = PROVEDOR
        except Exception as erro:
            print(f"⚠️ LLM indisponível ({erro}). Usando modo offline.")
            bruto = interpretar_offline(texto)
            fonte = "offline"
    origem = resolver_nome(bruto.get("origem"))
    destino = resolver_nome(bruto.get("destino"))
    if origem is None or destino is None:
        return None, f"Não entendi origem/destino (resposta bruta: {bruto})"
    pedido = {"origem": origem, "destino": destino,
              "acessibilidade": bool(bruto.get("acessibilidade"))}
    return pedido, f"Interpretado via {fonte}"

In [ ]:
# Célula 16 — testando o intérprete (o último caso deve ser REJEITADO)
pedidos_teste = [
    "Estou na Catedral da Sé e quero ir ao Neo Química Arena",
    "to no masp, bora pro theatro municipal, tô de cadeira de rodas",
    "Preciso sair da Vila Madalena e chegar na Vila Prudente",
    "Quero ir da Vila Madalena até a Avenida Paulista",
]
for texto in pedidos_teste:
    pedido, mensagem = interpretar_pedido(texto)
    print("📝", texto)
    print("   ", mensagem, "→", pedido, "\n")

⚠️ LLM indisponível (The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable). Usando modo offline.
📝 Estou na Catedral da Sé e quero ir ao Neo Química Arena
    Interpretado via offline → {'origem': ('local', 'Catedral da Sé'), 'destino': ('local', 'Neo Química Arena'), 'acessibilidade': False} 

⚠️ LLM indisponível (The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable). Usando modo offline.
📝 to no masp, bora pro theatro municipal, tô de cadeira de rodas
    Interpretado via offline → {'origem': ('local', 'MASP'), 'destino': ('local', 'Theatro Municipal'), 'acessibilidade': True} 

⚠️ LLM indisponível (The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable). Usando modo offline.
📝 Preciso sair da Vila Madalena e chegar na Vila Prudente
    Interpretado via offli

In [ ]:
# Célula 17 — narrador: explica a rota citando as baldeações, sem inventar
def narrar_offline(r):
    if r["caminho"] is None:
        return (f"Não existe rota de {r['origem']} até {r['destino']} "
                f"com as estações bloqueadas: {', '.join(r['bloqueadas']) or 'nenhuma listada — a rede ficou desconectada nesse trecho'}.")
    texto = (f"Embarque em {r['origem']} e siga até {r['destino']}: "
             f"{r['paradas']} parada(s), {r['baldeacoes']} baldeação(ões), cerca de {r['tempo_min']} minutos.")
    if r["pontos_baldeacao"]:
        trocas = "; ".join(f"em {e}, troque para a {l}" for e, l in r["pontos_baldeacao"])
        texto += " Trocas de linha: " + trocas + "."
    if r["alertas"]:
        texto += " Atenção: " + "; ".join(r["alertas"]) + " (elevador em manutenção)."
    return texto

PROMPT_NARRADOR = """Você é o NARRADOR do MetrôBot SP. Explique a rota ao passageiro
em português, em no máximo 5 frases curtas e simpáticas.
Use SOMENTE os dados do JSON. Não invente horários, linhas, estações ou atrações.
Se "caminho" for null, explique que não há rota e cite as estações bloqueadas.
Se houver baldeações, cite EXATAMENTE onde trocar de linha (campo "pontos_baldeacao").
Se houver "alertas", destaque-os."""

def narrar(resultado):
    if PROVEDOR == "offline":
        return narrar_offline(resultado)
    dados = {k: resultado[k] for k in
             ("origem", "destino", "caminho", "paradas", "baldeacoes",
              "pontos_baldeacao", "tempo_min", "bloqueadas", "alertas")}
    try:
        return chamar_llm([{"role": "system", "content": PROMPT_NARRADOR},
                            {"role": "user", "content": json.dumps(dados, ensure_ascii=False)}])
    except Exception as erro:
        return narrar_offline(resultado) + f" (narrador offline: {erro})"

In [ ]:
# Célula 18 — testando o narrador (rota com baldeação)
r = planejar({"origem": ("estacao", "Tucuruvi"), "destino": ("estacao", "Corinthians-Itaquera")})
print(narrar(r))

Embarque em Tucuruvi e siga até Corinthians-Itaquera: 22 parada(s), 1 baldeação(ões), cerca de 49 minutos. Trocas de linha: em Sé, troque para a Linha 3-Vermelha. (narrador offline: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable)


## R5 — Interface com ipywidgets

In [ ]:
# Célula 19 — desenho das 3 linhas, com as cores oficiais (CORES)
def desenhar_linhas(resultado):
    caminho = set(resultado["caminho"] or [])
    visitados = set(resultado["visitados"])
    bloqueadas = set(resultado["bloqueadas"])
    pontos_baldeacao = {e for e, _ in resultado.get("pontos_baldeacao", [])}
    blocos = []
    for nome_linha, estacoes in LINHAS.items():
        cor_linha = CORES[nome_linha]
        linhas_html = [f"<h4 style='color:{cor_linha};margin-bottom:4px'>{nome_linha}</h4>"]
        for estacao in estacoes:
            if estacao in bloqueadas:
                cor, marca = "#222222", "🚫 bloqueada"
            elif estacao in pontos_baldeacao:
                cor, marca = "#f9a825", "🔁 baldeação"
            elif estacao in (resultado["origem"], resultado["destino"]) and estacao in caminho:
                cor, marca = cor_linha, "⭐ " + ("origem" if estacao == resultado["origem"] else "destino")
            elif estacao in caminho:
                cor, marca = cor_linha, "rota"
            elif estacao in visitados:
                cor, marca = "#bdbdbd", "visitada pela busca"
            else:
                cor, marca = "#e0e0e0", ""
            linhas_html.append(
                f"<div style='display:flex;gap:8px;font-family:sans-serif;font-size:12px'>"
                f"<span style='width:12px;height:12px;border-radius:50%;background:{cor};display:inline-block'></span>"
                f"<span style='min-width:150px'>{estacao}</span><span style='color:#666'>{marca}</span></div>")
        blocos.append(f"<div style='border-left:4px solid {cor_linha};padding-left:8px;min-width:230px'>"
                       + "".join(linhas_html) + "</div>")
    return "<div style='display:flex;gap:24px;flex-wrap:wrap'>" + "".join(blocos) + "</div>"

In [ ]:
# Célula 20 — painel interativo com ipywidgets
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

opcoes = ([(f"📍 {local}", ("local", local)) for local in LOCAIS] +
          [(f"🚇 {estacao}", ("estacao", estacao)) for estacao in TODAS_ESTACOES])

txt_pedido = widgets.Textarea(
    placeholder="Ex.: Estou na Catedral da Sé e quero ir ao Neo Química Arena",
    layout=widgets.Layout(width="95%", height="60px"))
btn_interpretar = widgets.Button(description="💬 Interpretar pedido", button_style="info")
dd_origem = widgets.Dropdown(options=opcoes, description="Origem:")
dd_destino = widgets.Dropdown(options=opcoes, value=("estacao", "Jabaquara"), description="Destino:")
chk_acess = widgets.Checkbox(description="Preciso de acessibilidade")
sel_fechadas = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Fechadas:", rows=6)
sel_manut = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Elevador ⚙️:", rows=6)
sel_paralisadas = widgets.SelectMultiple(options=list(LINHAS.keys()), description="Linha parada:", rows=3)
rb_algoritmo = widgets.RadioButtons(options=["BFS", "DFS"], description="Busca:")
btn_buscar = widgets.Button(description="🚇 Buscar rota", button_style="success")
saida = widgets.Output()

def ao_interpretar(_):
    with saida:
        clear_output()
        pedido, msg = interpretar_pedido(txt_pedido.value)
        print(msg)
        if pedido:
            dd_origem.value = pedido["origem"]
            dd_destino.value = pedido["destino"]
            chk_acess.value = pedido["acessibilidade"]
            print("✅ Campos preenchidos. Confira e clique em 'Buscar rota'.")

def ao_buscar(_):
    with saida:
        clear_output()
        pedido = {"origem": dd_origem.value, "destino": dd_destino.value,
                   "acessibilidade": chk_acess.value}
        r = planejar(pedido, sel_fechadas.value, sel_manut.value,
                     sel_paralisadas.value, rb_algoritmo.value)
        display(HTML(f"<h4>{r['algoritmo']}: {r['origem']} → {r['destino']}</h4>"))
        print("🦙", narrar(r))
        if r["caminho"]:
            print(f"🔎 Paradas: {r['paradas']} | Baldeações: {r['baldeacoes']} | Tempo estimado: {r['tempo_min']} min")
            if r["pontos_baldeacao"]:
                print("🔁 Troque de linha em:", "; ".join(f"{e} → {l}" for e, l in r["pontos_baldeacao"]))
        print(f"🔍 Estações visitadas pela busca: {len(r['visitados'])}")
        print(f"📜 Regras disparadas: {', '.join(r['regras_usadas'])}")
        display(HTML(desenhar_linhas(r)))

btn_interpretar.on_click(ao_interpretar)
btn_buscar.on_click(ao_buscar)

painel = widgets.VBox([
    widgets.HTML("<h3>🚇 MetrôBot SP — Linhas 1, 2 e 3</h3>"),
    txt_pedido, btn_interpretar,
    widgets.HBox([dd_origem, dd_destino]),
    widgets.HBox([chk_acess, rb_algoritmo]),
    widgets.HBox([sel_fechadas, sel_manut, sel_paralisadas]),
    btn_buscar, saida,
])

In [ ]:
# Célula 21 — mostrar o app
display(painel)

## R6 — Testes automatizados

Os 6 casos obrigatórios do enunciado + 4 testes extras (grafo, integrações via R6, linha paralisada via R7 e locais conhecidos).

In [ ]:
# Célula 22 — testes automatizados
def rodar_testes():
    # 1. Grafo com 52 estações, sem vizinhos duplicados
    assert len(GRAFO) == 52, f"esperado 52, veio {len(GRAFO)}"
    for est, vizinhas in GRAFO.items():
        assert len(vizinhas) == len(set(vizinhas)), f"vizinhos duplicados em {est}"

    # 2. Caso obrigatório 1: Tucuruvi → Corinthians-Itaquera (normal)
    r = planejar({"origem": ("estacao", "Tucuruvi"), "destino": ("estacao", "Corinthians-Itaquera")})
    assert r["paradas"] == 22, f"caso1 paradas={r['paradas']}"
    assert r["baldeacoes"] == 1, f"caso1 baldeacoes={r['baldeacoes']}"

    # 3. Caso obrigatório 2: Vila Madalena → Jabaquara (normal)
    r = planejar({"origem": ("estacao", "Vila Madalena"), "destino": ("estacao", "Jabaquara")})
    assert r["paradas"] == 14, f"caso2 paradas={r['paradas']}"
    assert r["baldeacoes"] == 1, f"caso2 baldeacoes={r['baldeacoes']}"

    # 4. Caso obrigatório 3: Palmeiras-Barra Funda → Vila Prudente (normal)
    r = planejar({"origem": ("estacao", "Palmeiras-Barra Funda"), "destino": ("estacao", "Vila Prudente")})
    assert r["paradas"] == 16, f"caso3 paradas={r['paradas']}"
    assert r["baldeacoes"] == 2, f"caso3 baldeacoes={r['baldeacoes']}"

    # 5. Caso obrigatório 4: Tucuruvi → Brás, Sé fechada → sem rota
    r = planejar({"origem": ("estacao", "Tucuruvi"), "destino": ("estacao", "Brás")}, fechadas={"Sé"})
    assert r["caminho"] is None, "caso4 deveria ser sem rota"

    # 6. Caso obrigatório 5: Vila Madalena → Jabaquara, Paraíso fechada → sem rota (Linha 2 cortada)
    r = planejar({"origem": ("estacao", "Vila Madalena"), "destino": ("estacao", "Jabaquara")}, fechadas={"Paraíso"})
    assert r["caminho"] is None, "caso5 deveria ser sem rota"

    # 7. Caso obrigatório 6: Vila Prudente → Jabaquara, Paraíso fechada → 13 paradas via Ana Rosa (desvio)
    r = planejar({"origem": ("estacao", "Vila Prudente"), "destino": ("estacao", "Jabaquara")}, fechadas={"Paraíso"})
    assert r["paradas"] == 13, f"caso6 paradas={r['paradas']}"
    assert r["caminho"] is not None

    # 8. R6: integrações deduzidas automaticamente (Sé, Paraíso, Ana Rosa)
    fatos_teste, _ = encadear_para_frente(fatos_base(), REGRAS)
    integracoes = {e for (e,) in consultar(fatos_teste, "integracao")}
    assert integracoes == {"Sé", "Paraíso", "Ana Rosa"}, f"integracoes={integracoes}"

    # 9. R7: linha inteira paralisada bloqueia todas as suas estações
    r = planejar({"origem": ("estacao", "Palmeiras-Barra Funda"), "destino": ("estacao", "Corinthians-Itaquera")},
                  linhas_paralisadas=["Linha 3-Vermelha"])
    assert r["caminho"] is None, "linha paralisada deveria bloquear a rota"

    # 10. Locais conhecidos funcionam nas 3 linhas
    r_local = planejar({"origem": ("local", "Catedral da Sé"), "destino": ("local", "Neo Química Arena")})
    assert r_local["origem"] == "Sé" and r_local["destino"] == "Corinthians-Itaquera"

    print("✅ Todos os 10 testes passaram!")

rodar_testes()

✅ Todos os 10 testes passaram!


## Explicando os casos 5 e 6 (por que um tem rota e o outro não?)



 No caso 5, a origem (Vila Madalena) fica **antes** de Paraíso na lista da Linha 2-Verde. Como os trechos são sequenciais (Brigadeiro–Paraíso–Ana Rosa), fechar Paraíso corta o único caminho que liga o lado de Vila Madalena ao lado de Ana Rosa/Chácara Klabin/…/Vila Prudente — e é só a partir de Ana Rosa que existe conexão com a Linha 1. Por isso a Linha 2 fica "cortada em dois pedaços" e não há rota.

No caso 6, a origem (Vila Prudente) já está do **outro lado** de Paraíso (depois de Ana Rosa). Para chegar a Ana Rosa, o passageiro não precisa passar por Paraíso — ele pode ir na direção contrária (Vila Prudente → Tamanduateí → Sacomã → ... → Ana Rosa) e ali fazer baldeação para a Linha 1 até Jabaquara. Por isso existe um desvio de 13 paradas.

## Declaração de uso de IA

Utilizamos ferramentas de IA como apoio ao longo do desenvolvimento do projeto, principalmente para ajudar na organização de partes do código, revisar a lógica e identificar possíveis falhas. As sugestões geradas foram ajustadas de acordo com os requisitos do desafio e conferidas pelo grupo. Durante os testes, também fizemos correções relacionadas às rotas, às baldeações e à forma como o narrador apresentava as informações. Ao final, validamos o funcionamento do projeto com base nos casos de teste propostos no enunciado.
